# Approve a tool call before it runs

Let the agent propose a file edit, then decide whether to allow it. The file stays unchanged until you approve.

Run the cells in order. You need an [OpenAI API key](https://platform.openai.com/api-keys) with API credit.

[Open in Colab](https://colab.research.google.com/github/BerriAI/liteagents/blob/main/cookbook/recipes/04_approvals.ipynb)

## 1. Install

Install LiteAgents and the integrations used in this notebook.

In [ ]:
%pip install -q --progress-bar off "liteagents[pydantic-ai] @ https://github.com/BerriAI/liteagents/releases/download/v0.3.0a5/liteagents-0.3.0a5-py3-none-any.whl"

## 2. Add your key

Run this cell, paste your key into the hidden input, and press Enter.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = (os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")).strip()
if not os.environ["OPENAI_API_KEY"]:
    raise ValueError("Run this cell again and enter your OpenAI API key.")

## 3. Create the file

This is a temporary demo file with a pending status.

In [ ]:
import tempfile
from pathlib import Path

workspace = Path(tempfile.mkdtemp(prefix="liteagents-"))
target = workspace / "status.txt"
target.write_text("status=pending\n")
print(target.read_text())

## 4. Require approval for edits

`interrupt_on` pauses calls to `edit_file`. Reading the file can still happen immediately.

In [ ]:
from liteagents import LiteAgentClient, LiteAgentOptions, ProfileOptions

profile = ProfileOptions(
    harness="pydantic-ai",
    model="openai/gpt-5.4-mini",
)
profile.tools = ["read_file", "edit_file"]
profile.harness_options["interrupt_on"] = {"edit_file": True}

## 5. Review the proposed change

The next cell shows the edit and asks **Approve?** Enter `y` to apply it or `n` to cancel the run.

In [ ]:
target.write_text("status=pending\n")
async with LiteAgentClient(options=LiteAgentOptions(profile=profile, cwd=workspace)) as agent:
    handle = await agent.start_run(
        "Read status.txt, then replace status=pending with status=approved using edit_file."
    )
    approved = False
    async for event in handle.events():
        if event.kind == "approval_requested":
            print("Proposed edit:", event.data["arguments"])
            print("File before approval:", target.read_text().strip())
            if input("Approve? [y/N] ").strip().lower() != "y":
                await handle.cancel()
                print("Cancelled; the file was not changed.")
                break
            await handle.approve(event.data["id"])
            approved = True
    if approved:
        print((await handle.result()).text)

print("File after:", target.read_text().strip())

Run the last cell again and try the other answer. It resets only this demo file before starting.

The client closes when the cell finishes or is interrupted, cancelling unfinished local work.

[Other model providers](https://github.com/BerriAI/liteagents/blob/main/docs/models.md) · [All cookbooks](https://github.com/BerriAI/liteagents/blob/main/cookbook/README.md)